# LCA mini | 평가계수 전처리

Colab에서 수행한 데이터 정제 과정을 재현하기 위한 정리본입니다. 원본 데이터의 이용 조건을 확인하고 직접 내려받아 경로를 설정하세요.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

path = Path("/content/drive/MyDrive/LCA mini/한국환경산업기술원_환경성적표지 평가계수_20210802.csv")
for enc in ["utf-8-sig", "cp949", "euc-kr", "utf-8"]:
    try:
        df = pd.read_csv(path, encoding=enc)
        print("인코딩:", enc)
        break
    except UnicodeDecodeError:
        continue
else:
    raise ValueError("지원하는 인코딩으로 읽지 못했습니다.")

In [ ]:
df = df.replace("해당없음", np.nan)
impact_cols = [c for c in df.columns if "(" in c and c != "단위"]
df[impact_cols] = df[impact_cols].apply(pd.to_numeric, errors="coerce")
print("전체 행:", len(df))
print("대분류:\n", df["대분류"].value_counts())
print("구분:\n", df["구분"].value_counts())
print("단위:\n", df["단위"].value_counts())
print("결측 비율:\n", df[impact_cols].isna().mean().round(2))

In [ ]:
key = ["대분류", "구분", "원료_에너지", "단위"]
print("원료명 중복:", df["원료_에너지"].duplicated().sum())
print("복합키 중복:", df.duplicated(key).sum())
display(df[df.duplicated("원료_에너지", keep=False)].sort_values("원료_에너지").head(20))
print(df.groupby("대분류")["단위"].agg(lambda s: s.value_counts().head(5).to_dict()))

In [ ]:
c = ["탄소발자국(kg CO2 eq)", "자원발자국(kg Sb eq)", "산성비(kg SO2 eq)", "부영양화(kg PO43- eq)", "물소모계수(m3 H2O eq)"]
print("5개 모두 있는 행:", df[c].notna().all(axis=1).sum())
print("앞 3개 모두 있는 행:", df[c[:3]].notna().all(axis=1).sum())
out = Path("/content/drive/MyDrive/lca_factors_clean.csv")
df.to_csv(out, index=False, encoding="utf-8-sig")
print("저장:", out)